In [ ]:
from figure_paths import DATA, RESULTS, WORK, FIGURES, setting
TASK2_ROOT=RESULTS / 'task2_diagnostics'
NC_ROOT = TASK2_ROOT / 'mechanism_v4_20261028'
DPI = 800
SHOW_FIGURES = True

SOURCE_MODE = 'nc'                   # nc / auto / tables / memory
SOURCE_TABLE_DIR = None
FILES = {
    'pinn': NC_ROOT / 'pinn/pinn_paired_seed_mechanism_final.nc',
    'gradient': NC_ROOT / 'gradient/gradient_mechanism_v4.nc',
    'vmd': NC_ROOT / 'vmd/VMD_mechanism_timescale.nc',
}

MEMORY_TABLES = None

VARIABLE_OVERRIDES = {}             # 角色名: ('group/path', 'actual_variable')
DIMENSION_OVERRIDES = {'pinn': {}, 'gradient': {}, 'vmd': {}}
VALUE_OVERRIDES = {'pinn': {}, 'gradient': {}, 'vmd': {}}
SELECT_OVERRIDES = {
    'pinn': {'support': 'above_ground_full_stencil'},
    'gradient': {}, 'vmd': {},
}
UNIT_MULTIPLIERS = {}               # 正式 NC 已用物理单位；omega 不再乘 -10。

CONFIG = {
    'SOURCE_MODE': SOURCE_MODE, 'SOURCE_TABLE_DIR': SOURCE_TABLE_DIR,
    'TASK2_ROOT': TASK2_ROOT, 'NC_ROOT': NC_ROOT, 'FILES': FILES,
    'VARIABLE_OVERRIDES': VARIABLE_OVERRIDES, 'DIMENSION_OVERRIDES': DIMENSION_OVERRIDES,
    'VALUE_OVERRIDES': VALUE_OVERRIDES, 'SELECT_OVERRIDES': SELECT_OVERRIDES,
    'UNIT_MULTIPLIERS': UNIT_MULTIPLIERS, 'REQUIRE_FIRST_STAGE_GRADIENT': False,
    'PINN_BLOCK_HOURS': 72, 'GRADIENT_BLOCK_HOURS': 72, 'ALLOW_MISSING_CI': False,
}


In [ ]:
"""Read existing mechanism summaries for replacement figures, without recomputation.

Interface: load_plot_data(config, memory=None) -> pinn, gradient, vmd, audit.
The three frames retain the unrounded table schemas from the previous notebook.
"""
from pathlib import Path
from itertools import product
import hashlib
import json
import os
import re
import warnings
import numpy as np
import pandas as pd
import xarray as xr


class SchemaError(ValueError):
    pass


def norm(value):
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return re.sub(r'[^a-z0-9]+', '', str(value).lower().replace('ω', 'omega'))


DIM_ALIASES = {
    'lead': ['lead', 'lead_time', 'lead_hour', 'lead_hours', 'forecast_hour', 'forecast_hours', 'forecast_time', 'horizon'],
    'level': ['level', 'pressure', 'pressure_level', 'pressure_hpa', 'level_hpa', 'plev'],
    'seed': ['seed', 'random_seed', 'training_seed'],
    'component': ['component', 'wind_component', 'variable', 'output_component'],
    'stage': ['stage', 'model_stage', 'prediction_stage'],
    'scale': ['scale', 'spatial_scale', 'field_scale'],
    'condition': ['condition', 'condition_group', 'regime'],
    'condition_type': ['condition_type', 'condition_family', 'group_type', 'contrast_type', 'regime_axis'],
    'region': ['region', 'region_name', 'spatial_region', 'domain'],
    'mask': ['mask', 'mask_type', 'mask_variant', 'valid_domain', 'terrain_mask'],
    'support': ['support'],
    'operator': ['operator', 'operator_variant', 'derivative_operator', 'discretization'],
    'metric': ['metric', 'metric_name', 'score'],
    'term': ['term', 'decomposition_term', 'contribution', 'contribution_term', 'continuity_term', 'error_term'],
    'band': ['band', 'frequency_band', 'period_band', 'timescale', 'time_scale'],
    'block': ['block', 'block_hours', 'block_length', 'bootstrap_block_hours', 'block_length_hours'],
    'bound': ['bound', 'ci_bound', 'quantile', 'percentile', 'interval_bound', 'ci_side'],
}
ALIASES = {
    'stage': {'first': ['first', 'stage1', 'stage_1', 'first_stage', 'direct', 's1', 'stage1_main_forecast', 'stage1_main_prediction', 'stage1_forecast', '1'],
              'final': ['final', 'stage2', 'stage_2', 'final_corrected', 'residual_corrected', 'final_stage', 's2', 'final_forecast', 'stage2_final_forecast', 'stage2_corrected_forecast', 'stage2_final_prediction', '2']},
    'component': {'u': ['u', 'zonal', 'zonal_wind'], 'v': ['v', 'meridional', 'meridional_wind'], 'w': ['w', 'omega', 'pressure_vertical_velocity']},
    'scale': {'raw': ['raw', 'original', 'original_field', 'full', 'unsmoothed'],
              'smooth3': ['smooth3', 'smooth_3x3', 'smoothed_3x3', '3x3', 'smooth', 'smoothed', 'coarse', 'box3_smooth'],
              'fine': ['fine', 'fine_scale', 'fine_scale_residual', 'residual', 'highpass', 'high_pass', 'detail', 'box3_remainder']},
    'term': {'H': ['H', 'H2', 'eH2', 'E_eH2', 'mse_h', 'horizontal', 'horizontal_error_mse', 'horizontal_error_squared', 'horizontal_mse'],
             'Z': ['Z', 'Z2', 'eZ2', 'E_eZ2', 'mse_z', 'vertical', 'vertical_error_mse', 'vertical_error_squared', 'vertical_mse'],
             'cross': ['cross', 'cross_term', '2HZ', '2_eH_eZ', '2E_eHeZ', 'signed_cross_term', 'twice_cross', 'twice_cross_error'],
             'total': ['total', 'C', 'C2', 'eC2', 'E_eC2', 'mse_c', 'continuity', 'total_continuity_error_mse', 'continuity_mse']},
    'condition': {'all': ['all', 'all_conditions', 'overall', 'unconditional']},
    'condition_type': {'spatial': ['spatial', 'spatial_gradient', 'input_spatial_gradient', 'spatial_gradient_magnitude']},
    'region': {'all': ['all', 'full', 'full_region', 'whole_domain', 'all_region', 'full_domain', 'domain_all']},
    'mask': {'stencil': ['above_ground_complete_stencil', 'above_ground_full_stencil', 'above_ground_stencil', 'complete_stencil_above_ground', 'terrain_valid_stencil', 'above_ground_stencil_valid']},
    'support': {'stencil': ['above_ground_full_stencil', 'above_ground_complete_stencil']},
    'operator': {'training': ['training', 'original', 'original_training', 'training_operator', 'original_training_operator', 'native']},
    'metric': {'rmse': ['rmse', 'root_mean_square_error'], 'mse': ['mse', 'mean_square_error', 'mean_squared_error']},
    'bound': {'lower': ['lower', 'lo', 'low', 'lcl', 'lower95', 'ci_lower', 'p025', 'q025'],
              'upper': ['upper', 'hi', 'high', 'ucl', 'upper95', 'ci_upper', 'p975', 'q975']},
    'band': {'bias': ['bias', 'mean', 'mean_bias', 'bias2', 'bias_squared', 'squared_bias', 'dc'],
             '2–8 h': ['2-8h', '2_8h', '2to8h', '2-8', '2_8', '2–8 h'],
             '8–24 h': ['8-24h', '8_24h', '8to24h', '8-24', '8_24', '8–24 h'],
             '24–72 h': ['24-72h', '24_72h', '24to72h', '24-72', '24_72', '24–72 h'],
             '72–168 h': ['72-168h', '72_168h', '72to168h', '72-168', '72_168', '72–168 h'],
             '>168 h': ['>168h', 'gt168h', 'over168h', 'longer168h', '168hplus', '168-inf', '168inf', '>168 h'],
             '24–168 h': ['24-168h', '24_168h', '24to168h', '24-168', '24_168', '24–168 h'],
             '168–720 h': ['168-720h', '168_720h', '168to720h', '168-720', '168_720', '168–720 h'],
             '>720 h': ['>720h', 'gt720h', 'over720h', 'longer720h', '720hplus', '720-inf', '720inf', '>720 h']},
}
TOTAL_BANDS = {'total', 'all', 'overall', 'totalmse', 'allfrequencies'}
VARIABLES = {
    'pinn_delta': ('pinn', ['delta_continuity_error_contribution']),
    'pinn_ci': ('pinn', ['delta_continuity_error_contribution_ci', 'delta_continuity_error_contribution_CI', 'delta_continuity_error_contribution_ci95']),
    'pinn_total': ('pinn', ['delta_continuity_error_mse', 'delta_total_continuity_error_mse']),
    'pinn_total_ci': ('pinn', ['delta_continuity_error_mse_ci', 'delta_total_continuity_error_mse_ci']),
    'gradient_stage_change': ('gradient', ['rmse_gain_change_stage2_minus_stage1']),
    'gradient_stage_change_ci': ('gradient', ['rmse_gain_change_ci_raw_bound_expanded']),
    'gradient_gain': ('gradient', ['gain']),
    'gradient_gain_ci': ('gradient', ['stage_rmse_gain_ci_raw_bound_expanded']),
    'gradient_condition': ('gradient', ['rmse_gain_condition_high_minus_low']),
    'gradient_condition_ci': ('gradient', ['rmse_gain_condition_high_minus_low_ci_raw_bound_expanded', 'rmse_gain_condition_ci_raw_bound_expanded']),
    'gradient_condition_ci_saved': ('gradient', ['rmse_gain_condition_high_minus_low_ci']),
    'gradient_condition_bound': ('gradient', ['rmse_gain_condition_high_minus_low_raw_truth_bound']),
    'gradient_condition_bootstrap': ('gradient', ['rmse_gain_bootstrap']),
    'vmd_gain': ('vmd', ['vmd_mse_reduction']),
    'vmd_ci': ('vmd', ['vmd_mse_reduction_ci']),
}
NC_NAMES = {'pinn': 'pinn_paired_seed_mechanism_final.nc', 'gradient': 'gradient_mechanism_v4.nc', 'vmd': 'VMD_mechanism_timescale.nc'}
CSV_NAMES = {'pinn': 'pinn_plot_values.csv', 'gradient': 'gradient_plot_values.csv', 'vmd': 'vmd_plot_values_by_level.csv'}
KEYS = {'pinn': ['seed', 'level', 'term'], 'gradient': ['lead', 'level', 'component', 'kind', 'scale', 'stage'], 'vmd': ['lead', 'stage', 'component', 'level', 'band']}
CIS = {'pinn': [('lower', 'upper')], 'gradient': [('lower', 'upper')], 'vmd': [('lo72', 'hi72'), ('lo168', 'hi168')]}


def _canonical(dim, value):
    hits = [key for key, labels in ALIASES.get(dim, {}).items() if norm(value) in {norm(x) for x in labels + [key]}]
    if not hits and dim == 'band':
        text = str(value).lower().strip().replace('–', '-').replace('—', '-').replace('hours', 'h').replace('hour', 'h')
        text = re.sub(r'\s+', '', text)
        interval = re.fullmatch(r'(\d+(?:\.\d+)?)h?(?:_|-|to)(\d+(?:\.\d+)?)h?', text)
        above = re.fullmatch(r'(?:>|gt|over)(\d+(?:\.\d+)?)h?', text)
        if interval and float(interval.group(1)) < float(interval.group(2)):
            return f'{float(interval.group(1)):g}–{float(interval.group(2)):g} h'
        if above:
            return f'>{float(above.group(1)):g} h'
    if len(hits) != 1:
        raise SchemaError(f'{dim} 坐标 {value!r} 无法唯一识别；请用 VALUE_OVERRIDES 显式说明。')
    return hits[0]


def _number(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        text = str(value).lower().strip()
        match = re.fullmatch(r'(?:lead|seed|h)?\s*(\d+(?:\.\d+)?)\s*(?:h|hour|hours|hpa|pa)?', text)
        return float(match.group(1)) if match else np.nan


def _label(da, dim, target):
    vals = np.asarray(da[dim].values).reshape(-1).tolist()
    if dim in ('lead', 'level', 'seed', 'block'):
        nums = [_number(v) for v in vals]
        desired = float(target)
        if dim == 'level' and np.any(np.isfinite(nums)) and np.nanmedian(nums) > 2000:
            desired *= 100
        hits = [v for v, n in zip(vals, nums) if np.isfinite(n) and np.isclose(n, desired, rtol=0, atol=1e-6)]
    elif dim == 'band':
        hits = []
        for value in vals:
            if norm(value) in TOTAL_BANDS:
                continue
            if _canonical('band', value) == _canonical('band', target):
                hits.append(value)
    elif dim == 'bound':
        wanted = [0.025, 2.5] if target == 'lower' else [0.975, 97.5]
        hits = []
        for v in vals:
            try:
                number = float(str(v).strip().rstrip('%'))
                ok = any(np.isclose(number, n, rtol=0, atol=1e-9) for n in wanted)
            except ValueError:
                ok = norm(v) in {norm(t) for t in ALIASES['bound'][target]}
            if ok:
                hits.append(v)
    else:
        allowed = {norm(v) for v in ALIASES.get(dim, {}).get(str(target), [target]) + [target]}
        hits = [v for v in vals if norm(v) in allowed]
    if len(hits) != 1:
        raise SchemaError(f'{da.name}: {dim}={target!r} 匹配 {len(hits)} 项；实际坐标 {vals}。')
    return hits[0]


def _select(da, targets, required=()):
    absent = [d for d in required if d not in da.dims and d not in da.coords]
    if absent:
        raise SchemaError(f'{da.name} 缺少必要坐标 {absent}；实际维度 {da.dims}。')
    for dim, target in targets.items():
        if dim in da.dims:
            da = da.sel({dim: _label(da, dim, target)}, drop=False)
        elif dim in da.coords:
            _label(da, dim, target)
    return da


def _scalar(da, role):
    if da.size != 1:
        raise SchemaError(f'{role} 尚有未选择维度 {dict(da.sizes)}；请指定 SELECT_OVERRIDES，不能自动平均或取首项。')
    return float(np.asarray(da.values).reshape(-1)[0]) * float(da.attrs.get('_plot_unit_multiplier', 1))


def _hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def _validate_frames(frames, require_first=False):
    """Require exactly the intended comparison cells, never broadcast missing slices."""
    result = {}
    gradient_specs = [('stage_change', 'raw', 'contrast'), ('gain', 'raw', 'final'), ('gain', 'smooth3', 'final'), ('gain', 'fine', 'final'), ('condition_contrast', 'raw', 'final')]
    gradient_input = frames['gradient']
    has_first = False
    if {'kind', 'stage'}.issubset(gradient_input.columns):
        has_first = any(_canonical('stage', s) == 'first' for s in gradient_input.loc[gradient_input['kind'] == 'gain', 'stage'])
    if require_first or has_first:
        gradient_specs += [('gain', s, 'first') for s in ('raw', 'smooth3', 'fine')]
    expected = {
        'pinn': set(product([42, 2026], [850, 925, 1000], ['H', 'Z', 'cross', 'total'])),
        'gradient': set((h, p, c, k, s, stage) for h, p, c in product(range(1, 7), [850, 925, 1000], ['u', 'v', 'w'])
                        for k, s, stage in gradient_specs),
        'vmd': None,
    }
    for family in CSV_NAMES:
        df = frames[family].copy(deep=True)
        required = KEYS[family] + ['value'] + [x for pair in CIS[family] for x in pair]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise SchemaError(f'{family} 表缺少列 {missing}。')
        for d in ('lead', 'level', 'seed'):
            if d in df:
                numbers = df[d].map(_number)
                if d == 'level' and numbers.median() > 2000:
                    numbers = numbers / 100
                if not np.isfinite(numbers).all() or not np.allclose(numbers, np.round(numbers)):
                    raise SchemaError(f'{family}.{d} 包含非整数或无法识别的坐标。')
                df[d] = numbers.astype(int)
        for d in ('term', 'component', 'scale', 'band', 'stage'):
            if d in df:
                df[d] = df[d].map(lambda x: 'contrast' if d == 'stage' and str(x) == 'contrast' else _canonical(d, x))
        if df.duplicated(KEYS[family]).any():
            raise SchemaError(f'{family} 表存在重复比较单元，不能重复累计。')
        if family == 'vmd':
            bands = list(dict.fromkeys(df['band']))
            if 'bias' not in bands or len(bands) < 3:
                raise SchemaError(f'VMD 频带集合不完整：{bands}。须包括 bias 和已交付频带。')
            expected['vmd'] = set(product(range(1, 7), ['first', 'final'], ['u', 'v', 'w'], [850, 925, 1000], bands))
        actual = set(df[KEYS[family]].itertuples(index=False, name=None))
        absent, extra = expected[family] - actual, actual - expected[family]
        if absent or extra:
            raise SchemaError(f'{family} 覆盖不匹配：缺少 {len(absent)} 项（例 {list(absent)[:3]}），多出 {len(extra)} 项（例 {list(extra)[:3]}）。')
        numeric = ['value'] + [x for pair in CIS[family] for x in pair]
        for c in numeric:
            df[c] = pd.to_numeric(df[c], errors='raise')
        if not np.isfinite(df['value']).all():
            raise SchemaError(f'{family} 点估计存在非有限值，请检查正式汇总。')
        for lo, hi in CIS[family]:
            valid_lo, valid_hi = np.isfinite(df[lo]), np.isfinite(df[hi])
            if (valid_lo != valid_hi).any():
                raise SchemaError(f'{family} {lo}/{hi} 存在单侧缺失的区间。')
            if (df.loc[valid_lo, lo] > df.loc[valid_hi, hi]).any():
                raise SchemaError(f'{family} {lo}/{hi} 上下限倒置。')
            df[f'ci_available_{lo}'] = valid_lo & valid_hi
        result[family] = df
    piv = result['pinn'].pivot(index=['seed', 'level'], columns='term', values='value')
    if not np.allclose(piv[['H', 'Z', 'cross']].sum(axis=1), piv['total'], rtol=2e-4, atol=1e-3):
        raise SchemaError('PINN 的 H + Z + cross 与 total 不闭合；请核对交叉项是否已包含因子 2。')
    return result


def _table_provenance(directory, explicit=False):
    audit_path = Path(directory) / 'plot_audit.json'
    if not audit_path.is_file():
        if explicit:
            return {'status': 'explicit_table_directory', 'note': '按用户显式选择使用旧绘图表；表中 PINN 已为 10^-12 s^-2，其余为 SI。原 NC 来源未由伴随审计核验。'}
        raise SchemaError('自动发现的表缺少 plot_audit.json。')
    audit = json.loads(audit_path.read_text(encoding='utf-8-sig'))
    selected = audit.get('selected', {})
    for family, expected_name in NC_NAMES.items():
        entries = [v for k, v in selected.items() if k.startswith(family + '_') and isinstance(v, dict)]
        if not entries:
            raise SchemaError(f'表的审计没有 {family} 变量来源。')
        paths = {str(e.get('path', '')).replace('\\', '/') for e in entries}
        if len(paths) != 1 or next(iter(paths)).split('/')[-1] != expected_name:
            raise SchemaError(f'{family} 表引用了不一致或非冻结 v4 来源：{paths}。')
    if '10^-12' not in str(audit.get('pinn_scope', '')) and '1e-12' not in str(audit.get('pinn_scope', '')).lower():
        raise SchemaError('PINN 表审计未声明 10^-12 s^-2 绘图单位，不能猜测缩放。')
    return {'status': 'matching_frozen_nc_audit', 'audit_path': str(audit_path), 'original_audit': audit}


def _shallow_table_dirs(root, max_depth=4):
    """Only inspect directory names and tiny CSV sets, not large raw/block trees."""
    root = Path(root)
    if not root.is_dir():
        return []
    found = []
    blocked = {'blocks', 'chunks', 'raw', 'raw_data', 'cache', 'caches', 'runtime', 'checkpoints', 'bootstrap', 'block_sufficient_statistics', 'intermediate', 'workers'}
    for base, dirs, files in os.walk(root):
        depth = len(Path(base).relative_to(root).parts)
        dirs[:] = [d for d in dirs if d.lower() not in blocked and not d.startswith('.') and not re.match(r'^(?:job|block|chunk)_', d.lower())] if depth < max_depth else []
        if set(CSV_NAMES.values()).issubset(files):
            found.append(Path(base))
    return found


def _exact_summary_candidates(roots, basename, max_depth=4):
    """Find only an exact official filename; prune large computation trees."""
    found = {}
    blocked = {'blocks', 'chunks', 'raw', 'raw_data', 'cache', 'caches', 'runtime',
               'checkpoints', 'bootstrap', 'block_sufficient_statistics', 'intermediate', 'workers'}
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for base, dirs, files in os.walk(root):
            depth = len(Path(base).relative_to(root).parts)
            dirs[:] = [d for d in dirs if d.lower() not in blocked and not d.startswith('.')
                       and not re.match(r'^(?:job|block|chunk)_', d.lower())] if depth < max_depth else []
            if basename in files:
                candidate = (Path(base) / basename).resolve()
                found[str(candidate)] = candidate
    return list(found.values())


class _NCReader:
    def __init__(self, config, audit):
        self.config, self.audit = config, audit
        self.datasets, self.schemas, self.files = {}, {}, {}
        base = Path(config.get('NC_ROOT', Path(config.get('TASK2_ROOT', TASK2_ROOT)) / 'mechanism_v4_20261028'))
        supplied = config.get('FILES', {})
        overrides = config.get('FILE_OVERRIDES', {})
        task_root = Path(config.get('TASK2_ROOT', TASK2_ROOT))
        search_roots = [base, task_root / 'supplement_v2_20261028']
        resolution = {}
        for family, name in NC_NAMES.items():
            intended = Path(overrides.get(family) or supplied.get(family) or base / family / name)
            selected = intended
            entry = {'intended': str(intended), 'policy': 'explicit/default file exists'}
            if not intended.is_file():
                candidates = _exact_summary_candidates(search_roots, name)
                entry.update(policy='unique exact official basename within bounded search',
                             search_roots=[str(p) for p in search_roots],
                             candidates=[str(p) for p in candidates])
                if len(candidates) > 1:
                    self.audit.setdefault('nc_path_resolution', {})[family] = entry
                    raise SchemaError(f'{family} 原路径不存在，并找到多个同名正式 NC：{entry["candidates"]}。'
                                      f'请用 FILES["{family}"] 或 FILE_OVERRIDES["{family}"] 指定唯一文件；不按时间选择。')
                if candidates:
                    selected = candidates[0]
                else:
                    entry['status'] = 'not_found'
            entry['selected'] = str(selected)
            resolution[family] = entry
            self.files[family] = selected
        self.audit['nc_path_resolution'] = resolution
        self.audit['nc_files'] = {k: str(v) for k, v in self.files.items()}

    def close(self):
        for ds in self.datasets.values():
            ds.close()

    def schema(self, family):
        if family not in self.schemas:
            path = self.files[family]
            if not path.is_file():
                raise FileNotFoundError(f'{family} 正式 NC 不存在：{path}。可设置 SOURCE_TABLE_DIR 指向先前成功绘图产生的三个未舍入 CSV。')
            import netCDF4
            entries = []
            metadata = {}
            def visit(group, name):
                entries.extend((name, n) for n in group.variables)
                small = {}
                for n, var in group.variables.items():
                    if any(t in n.lower() for t in ('source_version', 'ci_availability_reason', 'seed_source', 'sample_count', 'n_samples')) and var.size <= 10000:
                        try:
                            small[n] = {'dims': list(var.dimensions), 'values': np.asarray(var[:]).tolist(), 'attrs': {a: var.getncattr(a) for a in var.ncattrs()}}
                        except (ValueError, TypeError):
                            pass
                metadata[name] = {'attrs': {a: group.getncattr(a) for a in group.ncattrs()}, 'provenance_variables': small}
                for n, child in group.groups.items():
                    visit(child, name.rstrip('/') + '/' + n)
            with netCDF4.Dataset(path) as root:
                visit(root, '/')
            self.schemas[family] = entries
            self.audit.setdefault('nc_provenance', {})[family] = metadata
        return self.schemas[family]

    def ds(self, family, group):
        key = (family, group)
        if key not in self.datasets:
            self.datasets[key] = xr.open_dataset(self.files[family], group=None if group == '/' else group, engine='netcdf4', decode_times=False, cache=False)
        return self.datasets[key]

    def _prepare(self, da, family, role, ds):
        overrides = self.config.get('DIMENSION_OVERRIDES', {}).get(family, {})
        rename = {}
        for d in dict.fromkeys(list(da.dims) + [c for c in da.coords if da[c].ndim == 0]):
            target = overrides.get(d)
            if target is None:
                vals = np.asarray(da[d].values).reshape(-1).tolist() if d in da.coords else []
                semantic = set()
                if family == 'pinn' and vals and len(vals) <= 30:
                    for value in vals:
                        try:
                            semantic.add(_canonical('term', value))
                        except SchemaError:
                            pass
                if family == 'vmd' and norm(d) == 'scale':
                    target = 'band'
                elif {'H', 'Z', 'cross'}.issubset(semantic):
                    target = 'term'
                else:
                    choices = [key for key, names in DIM_ALIASES.items() if norm(d) in {norm(x) for x in names}]
                    target = choices[0] if len(choices) == 1 else d
                    if target == d and vals and len(vals) == 2:
                        probe = xr.DataArray([0, 0], dims=['bound'], coords={'bound': vals})
                        try:
                            _label(probe, 'bound', 'lower'); _label(probe, 'bound', 'upper')
                            target = 'bound'
                        except SchemaError:
                            pass
            if target != d:
                if target in rename.values() or target in da.dims or target in da.coords:
                    raise SchemaError(f'{role} 重命名冲突 {d}->{target}。')
                rename[d] = target
        da = da.rename(rename)
        metadata = {**ds.attrs, **da.attrs}
        for d, aliases in DIM_ALIASES.items():
            if d in da.dims or d in da.coords:
                continue
            values = [v for k, v in metadata.items() if norm(k) in {norm(x) for x in aliases} and np.asarray(v).ndim == 0]
            if values and all(str(v) == str(values[0]) for v in values):
                da = da.assign_coords({d: values[0]})
        if family == 'pinn' and 'ci' in role and 'block' not in da.coords and 'block' not in da.dims:
            bootstrap = str(metadata.get('bootstrap', ''))
            hours = {float(v) for v in re.findall(r'(?<![\d.])(\d+(?:\.\d+)?)\s*h(?:our)?s?', bootstrap, flags=re.I)}
            if len(hours) == 1:
                fixed_hours = hours.pop()
                da = da.assign_coords(block=fixed_hours)
                self.audit.setdefault('fixed_ci_block_metadata', {})[role] = {'hours': fixed_hours, 'source_attribute': 'bootstrap', 'text': bootstrap}
        for d, mapping in self.config.get('VALUE_OVERRIDES', {}).get(family, {}).items():
            if d in da.dims:
                da = da.assign_coords({d: [mapping.get(v.item() if hasattr(v, 'item') else v, v) for v in da[d].values]})
            elif d in da.coords and da[d].ndim == 0:
                value = da[d].item(); da = da.assign_coords({d: mapping.get(value, value)})
        selectors = {**self.config.get('SELECT_OVERRIDES', {}).get(family, {}), **self.config.get('VARIABLE_SELECT_OVERRIDES', {}).get(role, {})}
        for d, v in selectors.items():
            d = rename.get(d, d)
            if d in da.dims:
                da = da.sel({d: v}, drop=False)
            elif d in da.coords:
                if str(da[d].item()) != str(v):
                    raise SchemaError(f'{role} 显式标量选择 {d}={v} 与文件不一致。')
            else:
                raise SchemaError(f'{role} 显式选择的 {d} 不在变量中。')
        units = str(da.attrs.get('units', '')).lower().replace(' ', '')
        if any(s in units for s in ['%', 'percent', 'normalized', 'standardized']):
            raise SchemaError(f'{role} 要求物理量，文件单位为 {units!r}。')
        multiplier = self.config.get('UNIT_MULTIPLIERS', {}).get(role)
        if multiplier is None:
            multiplier = 1e-12 if family == 'pinn' and any(t in units for t in ['1e-12', '10^-12', '10**-12', '10^{-12}']) else 1.0
        if not units:
            unit_name = 'physical_units' if family == 'gradient' else 'component_squared_error_units' if family == 'vmd' else None
            if unit_name and unit_name in ds and ds[unit_name].size <= 20:
                values = np.asarray(ds[unit_name].values).tolist()
                self.audit.setdefault('component_unit_evidence', {})[role] = {'variable': unit_name, 'units': values, 'extra_w_conversion': False}
            else:
                self.audit['warnings'].append(f'{role} 未提供 units；按正式机制汇总的已声明物理单位读取，未对 w 作额外换算。')
        return da.assign_attrs({**da.attrs, '_plot_unit_multiplier': float(multiplier)})

    def variable(self, role, optional=False, candidates=None):
        family, names = VARIABLES[role]
        names = candidates or names
        explicit = self.config.get('VARIABLE_OVERRIDES', {}).get(role)
        if explicit and candidates is None:
            if isinstance(explicit, dict):
                return explicit  # CI {'lower':(group,name), 'upper':(...)} handled by interval().
            matches = [('/' + str(explicit[0]).strip('/'), explicit[1])]
        else:
            matches = [(g, n) for g, n in self.schema(family) if n in names]
        if len(matches) == 0 and optional:
            return None
        if len(matches) != 1:
            raise SchemaError(f'{role} 需要唯一变量，实际 {matches}。请设置 VARIABLE_OVERRIDES[\'{role}\']=(group,variable)。')
        group, name = matches[0]
        ds = self.ds(family, group)
        self.audit['selected'][role + (':' + name if candidates else '')] = {'path': str(self.files[family]), 'group': group, 'variable': name, 'original_dims': list(ds[name].dims), 'attrs': dict(ds[name].attrs)}
        return self._prepare(ds[name], family, role, ds)

    def interval(self, role):
        family, names = VARIABLES[role]
        explicit = self.config.get('VARIABLE_OVERRIDES', {}).get(role)
        if isinstance(explicit, dict):
            endpoints = []
            for side in ('lower', 'upper'):
                if side not in explicit:
                    raise SchemaError(f'{role} 端点映射须同时含 lower/upper。')
                group, name = explicit[side]; group = '/' + str(group).strip('/')
                ds = self.ds(family, group)
                endpoints.append(self._prepare(ds[name], family, role, ds))
                self.audit['selected'][role + '_' + side] = {'path': str(self.files[family]), 'group': group, 'variable': name}
            return tuple(endpoints)
        packed = self.variable(role, optional=True)
        if packed is not None:
            return packed
        endpoint_arrays = []
        for side, endings in [('lower', ['lower', 'low', 'lo', 'lcl']), ('upper', ['upper', 'high', 'hi', 'ucl'])]:
            names_side = []
            for name in names:
                names_side.extend(name + '_' + end for end in endings)
                if name.endswith('_ci'):
                    names_side.extend(name[:-3] + '_' + end for end in endings)
            endpoint_arrays.append(self.variable(role, optional=True, candidates=names_side))
        if all(a is None for a in endpoint_arrays):
            self.audit['warnings'].append(f'{role} 未找到区间变量；保存点估计，区间保持缺失。')
            return None
        if any(a is None for a in endpoint_arrays):
            raise SchemaError(f'{role} 只找到一侧 CI。请明确 lower/upper 变量。')
        return tuple(endpoint_arrays)

    @staticmethod
    def common(da, family, block=None):
        targets = {'condition': 'all', 'region': 'all'}
        if family == 'pinn':
            targets.update(lead=1, stage='first', mask='stencil', support='stencil', operator='training')
        else:
            targets['metric'] = 'mse' if family == 'vmd' else 'rmse'
        if block is not None:
            targets['block'] = block
        return _select(da, targets)

    def bounds(self, ci, targets, family, role, block):
        if ci is None:
            return np.nan, np.nan
        needed = list(targets) + (['block'] if block is not None else [])
        try:
            def endpoint(da, side=None):
                selected = self.common(_select(da, targets, required=needed), family, block)
                if side is not None:
                    selected = _select(selected, {'bound': side}, required=['bound'])
                return _scalar(selected, role)
            if isinstance(ci, tuple):
                lo, hi = endpoint(ci[0]), endpoint(ci[1])
            else:
                lo, hi = endpoint(ci, 'lower'), endpoint(ci, 'upper')
            if np.isfinite(lo) != np.isfinite(hi) or (np.isfinite(lo) and lo > hi):
                raise SchemaError(f'{role} CI 缺单侧或倒置。')
            return lo, hi
        except SchemaError as exc:
            raise SchemaError(f'{role} 的已有 CI 坐标未正确匹配：{exc}') from exc

    def extract_gradient(self):
        rows = []
        specs = [('stage_change', 'gradient_stage_change', ['raw'], 'contrast'), ('gain', 'gradient_gain', ['raw', 'smooth3', 'fine'], 'final'), ('condition_contrast', 'gradient_condition', ['raw'], 'final')]
        if self.config.get('REQUIRE_FIRST_STAGE_GRADIENT', False):
            specs.insert(1, ('gain', 'gradient_gain', ['raw', 'smooth3', 'fine'], 'first'))
        for kind, role, scales, stage in specs:
            da, ci = self.variable(role), self.interval(role + '_ci')
            condition_bound = condition_bootstrap = None
            if kind == 'condition_contrast' and ci is None:
                saved_ci = self.interval('gradient_condition_ci_saved')
                condition_bound = self.variable('gradient_condition_bound')
                condition_bootstrap = self.variable('gradient_condition_bootstrap')
                ci = saved_ci
                self.audit['warnings'] = [m for m in self.audit['warnings'] if not m.startswith('gradient_condition_ci 未找到')]
                self.audit['gradient_condition_ci_assembly'] = {
                    'method': 'percentile(2.5,97.5) of saved paired spatial_high minus spatial_low RMSE gain draws, then minus/plus saved high-low raw-truth bound',
                    'bootstrap_variable': 'rmse_gain_bootstrap',
                    'comparison_ci_variable': 'rmse_gain_condition_high_minus_low_ci',
                    'bound_variable': 'rmse_gain_condition_high_minus_low_raw_truth_bound',
                    'new_bootstrap_draws': False,
                    'saved_ci_comparison': []}
            if kind == 'condition_contrast':
                def condition_slice(x):
                    if x is None:
                        return None
                    if isinstance(x, tuple):
                        return tuple(condition_slice(a) for a in x)
                    if 'condition_type' not in x.coords and 'condition' in x.coords:
                        x = x.rename({'condition': 'condition_type'})
                    return _select(x, {'condition_type': 'spatial'})
                da, ci, condition_bound = condition_slice(da), condition_slice(ci), condition_slice(condition_bound)
            da = self.common(da, 'gradient')
            for lead, level, comp, scale in product(range(1, 7), [850, 925, 1000], ['u', 'v', 'w'], scales):
                targets = dict(lead=lead, level=level, component=comp, scale=scale)
                if stage != 'contrast':
                    targets['stage'] = stage
                required = ['lead', 'level', 'component'] + (['stage', 'scale'] if kind == 'gain' else [])
                value = _scalar(_select(da, targets, required=required), role)
                paired_targets = {d: v for d, v in targets.items() if d in da.coords or d in da.dims}
                lo, hi = self.bounds(ci, paired_targets, 'gradient', role + '_ci', self.config.get('GRADIENT_BLOCK_HOURS', 72))
                if condition_bootstrap is not None:
                    bound = _scalar(_select(self.common(condition_bound, 'gradient'), paired_targets, required=paired_targets), 'gradient_condition_bound')
                    if not np.isfinite(bound) or bound < 0:
                        raise SchemaError('梯度条件真值扰动界限必须为有限非负数。')
                    boot_targets = {**paired_targets, 'block': self.config.get('GRADIENT_BLOCK_HOURS', 72)}
                    draws = _select(condition_bootstrap, boot_targets, required=list(boot_targets) + ['condition', 'replicate'])
                    high = _select(draws, {'condition': 'spatial_high'})
                    low = _select(draws, {'condition': 'spatial_low'})
                    if high.dims != ('replicate',) or low.dims != ('replicate',):
                        raise SchemaError(f'条件 bootstrap 选择后应只剩 replicate；实际 {high.dims}/{low.dims}。')
                    high, low = xr.align(high, low, join='exact')
                    delta_draws = np.asarray((high - low).values, dtype=float)
                    finite_draws = delta_draws[np.isfinite(delta_draws)]
                    if finite_draws.size < 2:
                        raise SchemaError('条件 bootstrap 的有效配对抽样不足，无法读取其百分位区间。')
                    raw_lo, raw_hi = np.percentile(finite_draws, [2.5, 97.5])
                    expanded = np.array([raw_lo - bound, raw_hi + bound])
                    saved = np.array([lo, hi])
                    matches_raw = np.allclose(saved, [raw_lo, raw_hi], rtol=1e-7, atol=1e-10)
                    matches_expanded = np.allclose(saved, expanded, rtol=1e-7, atol=1e-10)
                    comparison = 'both' if matches_raw and matches_expanded else 'raw' if matches_raw else 'expanded' if matches_expanded else 'different' if np.isfinite(saved).all() else 'unavailable'
                    self.audit['gradient_condition_ci_assembly']['saved_ci_comparison'].append({
                        'lead': lead, 'component': comp, 'level': level,
                        'paired_draws': int(delta_draws.size), 'finite_draws': int(finite_draws.size),
                        'raw_interval': [float(raw_lo), float(raw_hi)], 'truth_bound': float(bound),
                        'saved_interval': saved.tolist(), 'classification': comparison})
                    lo, hi = map(float, expanded)
                rows.append([lead, level, comp, kind, scale, stage, value, lo, hi])
        comparisons = self.audit.get('gradient_condition_ci_assembly', {}).get('saved_ci_comparison', [])
        different = sum(r['classification'] == 'different' for r in comparisons)
        if different:
            self.audit['warnings'].append(f'{different} 个保存的条件 CI 与现有配对抽样重建的 raw/expanded 端点均不一致；本图使用现有抽样重建后只扩界一次的区间，逐项差异已记录。')
        return pd.DataFrame(rows, columns=KEYS['gradient'] + ['value', 'lower', 'upper'])

    def extract(self):
        rows = []
        delta = self.common(self.variable('pinn_delta'), 'pinn')
        if 'term' not in delta.dims:
            raise SchemaError(f'PINN contribution 的实际坐标没有识别出 H/Z/cross：{list(delta.coords)}。请将实际分项维度映射到 term，并以 VALUE_OVERRIDES 提供标签。')
        ci = self.interval('pinn_ci')
        try:
            _label(delta, 'term', 'total'); has_total = True
        except SchemaError:
            has_total = False
        total_da = None if has_total else self.variable('pinn_total', optional=True)
        total_ci = None if has_total else self.interval('pinn_total_ci')
        for seed, level in product([42, 2026], [850, 925, 1000]):
            parts = []
            for term in ['H', 'Z', 'cross']:
                targets = dict(seed=seed, level=level, term=term)
                value = _scalar(_select(delta, targets, required=targets), 'pinn_delta')
                lo, hi = self.bounds(ci, targets, 'pinn', 'pinn_ci', self.config.get('PINN_BLOCK_HOURS', 72))
                parts.append(value); rows.append([seed, level, term, value * 1e12, lo * 1e12, hi * 1e12])
            targets = dict(seed=seed, level=level)
            if has_total:
                targets['term'] = 'total'
                value = _scalar(_select(delta, targets, required=targets), 'pinn_delta')
                lo, hi = self.bounds(ci, targets, 'pinn', 'pinn_ci', self.config.get('PINN_BLOCK_HOURS', 72))
            elif total_da is not None:
                value = _scalar(_select(self.common(total_da, 'pinn'), targets, required=targets), 'pinn_total')
                lo, hi = self.bounds(total_ci, targets, 'pinn', 'pinn_total_ci', self.config.get('PINN_BLOCK_HOURS', 72))
            else:
                value, lo, hi = sum(parts), np.nan, np.nan
                self.audit['warnings'].append('PINN total 点估计按 H + Z + cross 恒等式求和；没有构造 total CI。')
            rows.append([seed, level, 'total', value * 1e12, lo * 1e12, hi * 1e12])
        pinn = pd.DataFrame(rows, columns=KEYS['pinn'] + ['value', 'lower', 'upper'])
        gradient = self.extract_gradient()
        rows = []
        da, ci = self.common(self.variable('vmd_gain'), 'vmd'), self.interval('vmd_ci')
        if 'band' not in da.dims:
            raise SchemaError('VMD 的正式频带变量缺少 band 维度。')
        bands = [_canonical('band', b) for b in da['band'].values.tolist() if norm(b) not in TOTAL_BANDS]
        if len(bands) != len(set(bands)) or not bands:
            raise SchemaError(f'VMD 频带无法唯一选择：{bands}。')
        self.audit['vmd_bands'] = bands
        for lead, stage, comp, level, band in product(range(1, 7), ['first', 'final'], ['u', 'v', 'w'], [850, 925, 1000], bands):
            targets = dict(lead=lead, stage=stage, component=comp, level=level, band=band)
            value = _scalar(_select(da, targets, required=targets), 'vmd_gain')
            lo72, hi72 = self.bounds(ci, targets, 'vmd', 'vmd_ci', 72)
            lo168, hi168 = self.bounds(ci, targets, 'vmd', 'vmd_ci', 168)
            rows.append([lead, stage, comp, level, band, value, lo72, hi72, lo168, hi168])
        vmd = pd.DataFrame(rows, columns=KEYS['vmd'] + ['value', 'lo72', 'hi72', 'lo168', 'hi168'])
        return {'pinn': pinn, 'gradient': gradient, 'vmd': vmd}


def load_plot_data(config, memory=None):
    """Load one consistent bundle of previously computed formal summaries.

    SOURCE_MODE: 'auto' (default), 'tables', 'nc', or 'memory'.
    SOURCE_TABLE_DIR: explicit folder containing all three canonical CSV names.
    MEMORY: optional dict using pinn/gradient/vmd or pinn_plot/gradient_plot/vmd_plot.
    Auto-discovered CSVs require matching plot_audit.json; no newest-version rule.
    REQUIRE_FIRST_STAGE_GRADIENT=False preserves legacy 270-row table support.
    Set True only for an explicitly requested first-stage gain figure (432 rows).
    Missing confidence intervals remain NaN. No original observations are loaded.
    """
    config = dict(config)
    audit = {'selected': {}, 'warnings': [], 'source_mode': None,
             'contract': {'pinn': 'PINN minus NoPINN; 1 h; seeds 42/2026; 10^-12 s^-2',
                          'gradient': 'baseline minus augmented RMSE; m/s for u/v, Pa/s for w; expanded-reference CIs',
                          'vmd': 'baseline minus VMD MSE; fixed 25 locations; (m/s)^2 for u/v, (Pa/s)^2 for w',
                          'stage_change': 'final gain minus first-stage gain',
                          'ci_policy': 'Use saved intervals; if necessary reconstruct condition CI from existing paired draws and the saved truth bound once. Never average endpoints or construct cross-stage VMD intervals.'}}
    mode = str(config.get('SOURCE_MODE', 'auto')).lower()
    if mode not in ('auto', 'tables', 'nc', 'memory'):
        raise ValueError('SOURCE_MODE 必须是 auto/tables/nc/memory。')
    frames = None
    if memory is not None and mode in ('auto', 'memory'):
        frames = {f: memory.get(f, memory.get(f + '_plot')) for f in CSV_NAMES}
        if any(v is None for v in frames.values()):
            raise SchemaError('显式 memory 须同时提供 pinn、gradient、vmd 三份完整未舍入表。')
        audit.update(source_mode='memory', source_provenance='explicit caller-provided canonical plot frames')
    elif mode == 'memory':
        raise SchemaError('SOURCE_MODE=memory 但未传入 memory。')
    if frames is None and mode in ('auto', 'tables'):
        table_dir = config.get('SOURCE_TABLE_DIR')
        if table_dir:
            table_dir = Path(table_dir)
            proof = _table_provenance(table_dir, explicit=True)
            missing = [name for name in CSV_NAMES.values() if not (table_dir / name).is_file()]
            if missing:
                raise FileNotFoundError(f'{table_dir} 缺少 {missing}。')
        else:
            candidates = []
            for path in _shallow_table_dirs(config.get('TASK2_ROOT', TASK2_ROOT), config.get('TABLE_SEARCH_DEPTH', 4)):
                try:
                    proof_candidate = _table_provenance(path)
                    hashes = tuple(_hash(path / CSV_NAMES[f]) for f in CSV_NAMES)
                    candidates.append((path, proof_candidate, hashes))
                except (SchemaError, ValueError) as exc:
                    audit['warnings'].append(f'跳过自动表候选 {path}: {exc}')
            signatures = {entry[2] for entry in candidates}
            if len(signatures) == 1 and candidates:
                table_dir, proof, _ = sorted(candidates, key=lambda x: str(x[0]))[0]
                audit['equivalent_table_copies'] = [str(x[0]) for x in candidates]
            elif len(signatures) > 1:
                audit['warnings'].append('找到内容不同的合格表副本，未按时间选取；改读明确指定的冻结 NC。可用 SOURCE_TABLE_DIR 显式选择。')
                if mode == 'tables':
                    raise SchemaError('多份绘图表内容不同，请显式指定 SOURCE_TABLE_DIR。')
            if not candidates and mode == 'tables':
                raise FileNotFoundError('未找到有匹配来源审计的三份绘图表。请设置 SOURCE_TABLE_DIR。')
        if table_dir:
            frames = {f: pd.read_csv(table_dir / n) for f, n in CSV_NAMES.items()}
            audit.update(source_mode='tables', source_provenance=proof,
                         tables={f: {'path': str(table_dir / n), 'sha256': _hash(table_dir / n)} for f, n in CSV_NAMES.items()})
    if frames is None:
        reader = _NCReader(config, audit)
        try:
            frames = reader.extract()
        finally:
            reader.close()
        audit['source_mode'] = 'nc'
    grad = frames['gradient']
    stages_present = set()
    if {'stage', 'kind'}.issubset(grad.columns):
        stages_present = {_canonical('stage', v) for v in grad.loc[grad['kind'] == 'gain', 'stage']}
    if config.get('REQUIRE_FIRST_STAGE_GRADIENT', False) and 'first' not in stages_present:
        read_config = dict(config)
        if audit['source_mode'] == 'tables':
            provenance = audit.get('source_provenance', {})
            selected = provenance.get('original_audit', {}).get('selected', {})
            source_paths = {str(v['path']) for k, v in selected.items() if k.startswith('gradient_') and isinstance(v, dict) and 'path' in v}
            if len(source_paths) != 1:
                raise SchemaError('旧梯度表只有 final 阶段；新 Fig12 需要 first 阶段。请设 SOURCE_MODE="nc" 并提供冻结 gradient_mechanism_v4.nc；不能用 final 代替。')
            read_config['FILE_OVERRIDES'] = {**config.get('FILE_OVERRIDES', {}), 'gradient': source_paths.pop()}
        elif audit['source_mode'] == 'memory':
            raise SchemaError('传入的梯度表缺少 first 阶段收益；请设 SOURCE_MODE="nc" 从冻结 NC 读取两阶段完整表。')
        reader = _NCReader(read_config, audit)
        try:
            upgraded = reader.extract_gradient()
        finally:
            reader.close()
        keys = KEYS['gradient']
        check = grad.copy()
        for d in ('component', 'scale', 'stage'):
            check[d] = check[d].map(lambda v: 'contrast' if d == 'stage' and str(v) == 'contrast' else _canonical(d, v))
        shared = check.merge(upgraded, on=keys, suffixes=('_cached', '_nc'), validate='one_to_one')
        if len(shared) != len(check):
            raise SchemaError('旧梯度表与冻结 NC 的比较坐标不一致，未合并版本。请直接选择 SOURCE_MODE="nc"。')
        for col in ('value', 'lower', 'upper'):
            old, new = shared[col + '_cached'].to_numpy(float), shared[col + '_nc'].to_numpy(float)
            both = np.isfinite(old) & np.isfinite(new)
            if (np.isfinite(old) & ~np.isfinite(new)).any() or not np.allclose(old[both], new[both], rtol=1e-7, atol=1e-12):
                raise SchemaError(f'旧梯度表 {col} 与冻结 NC 的共有单元不一致，未混合结果版本。请核对来源或明确选择 SOURCE_MODE="nc"。')
        frames['gradient'] = upgraded
        audit['gradient_cache_upgrade'] = {'source': str(reader.files['gradient']), 'shared_rows_checked': len(shared), 'added_first_stage_rows': int(((upgraded['kind'] == 'gain') & (upgraded['stage'] == 'first')).sum())}
    frames = _validate_frames(frames, require_first=config.get('REQUIRE_FIRST_STAGE_GRADIENT', False))
    audit['vmd_bands'] = list(dict.fromkeys(frames['vmd']['band']))
    audit['coverage'] = {f: {'rows': len(df), 'finite_point_estimates': int(np.isfinite(df['value']).sum()),
                              'available_intervals': {lo + '/' + hi: int((np.isfinite(df[lo]) & np.isfinite(df[hi])).sum()) for lo, hi in CIS[f]}}
                         for f, df in frames.items()}
    audit['warnings'] = list(dict.fromkeys(audit['warnings']))
    for message in audit['warnings'][:8]:
        warnings.warn(message, stacklevel=2)
    if len(audit['warnings']) > 8:
        warnings.warn(f'另有 {len(audit["warnings"]) - 8} 条细节记录在返回的 audit["warnings"]。', stacklevel=2)
    return {**frames, 'audit': audit}


In [ ]:
"""Render frozen diagnostic summaries; never run model inference."""
import hashlib
import json
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from PIL import Image

LEVELS = [850, 925, 1000]
LEADS = [1, 2, 3, 4, 5, 6]
COMPONENTS = ['u', 'v', 'w']
SYMBOLS = {'u': r'$u$', 'v': r'$v$', 'w': r'$\omega$'}
STYLE = {
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size': 14, 'axes.labelsize': 14, 'axes.titlesize': 15,
    'xtick.labelsize': 12.5, 'ytick.labelsize': 12.5, 'legend.fontsize': 12,
    'axes.linewidth': .85, 'axes.unicode_minus': True,
    'figure.facecolor': 'white', 'savefig.facecolor': 'white',
    'mathtext.fontset': 'dejavusans',
}
CMAP = LinearSegmentedColormap.from_list(
    'pale_gain', ['#C99E78', '#F0DCC8', '#FFFFFF', '#D9EAF1', '#83B3C7'])
CMAP.set_bad('#EEEEEE')


def _unique(df, keys, name):
    if df.duplicated(keys).any():
        raise ValueError(f'{name}: duplicate coordinates; select the intended source before plotting.')


def _complete(df, keys, coordinates, name):
    _unique(df, keys, name)
    expected = pd.MultiIndex.from_product(coordinates, names=keys)
    actual = pd.MultiIndex.from_frame(df[keys])
    if len(actual) != len(expected) or len(expected.difference(actual)):
        raise ValueError(f'{name}: incomplete or unexpected coverage; expected {len(expected)}, got {len(actual)}.')
    if not np.isfinite(df['value'].to_numpy(float)).all():
        raise ValueError(f'{name}: nonfinite point estimates; no silent partial average is allowed.')


def _bounds(lo, hi):
    lo, hi = np.asarray(lo, float), np.asarray(hi, float)
    valid = np.isfinite(lo) & np.isfinite(hi)
    if np.any(lo[valid] > hi[valid]):
        raise ValueError('Saved confidence limits are reversed.')
    return valid


def prepare_figures(data):
    """Validate every panel before any existing image is replaced."""
    pinn, grad, vmd = [data[k].copy() for k in ('pinn', 'gradient', 'vmd')]
    for df, columns in [(pinn, ['lower', 'upper']), (grad, ['lower', 'upper']),
                        (vmd, ['lo72', 'hi72', 'lo168', 'hi168'])]:
        for col in columns:
            if col not in df:
                df[col] = np.nan
        for lo, hi in zip(columns[::2], columns[1::2]):
            _bounds(df[lo], df[hi])
    _complete(pinn, ['seed','level','term'], [[42,2026],LEVELS,['H','Z','cross','total']], 'PINN')
    wide = pinn.pivot(index=['seed','level'], columns='term', values='value')
    if not np.allclose(wide[['H','Z','cross']].sum(axis=1), wide['total'], rtol=2e-5, atol=1e-6):
        raise ValueError('PINN decomposition fails closure H + Z + cross = total in plot units.')
    stage = grad[(grad.kind == 'stage_change') & (grad.scale == 'raw')].copy()
    cond = grad[(grad.kind == 'condition_contrast') & (grad.scale == 'raw') & (grad.stage == 'final')].copy()
    scales = grad[(grad.kind == 'gain') & (grad.stage == 'final') & grad.scale.isin(['raw','smooth3','fine'])].copy()
    for name, sub in [('gradient stage change',stage),('gradient condition contrast',cond)]:
        _complete(sub, ['component','level','lead'], [COMPONENTS,LEVELS,LEADS], name)
    _complete(scales, ['scale','component','level','lead'], [['raw','smooth3','fine'],COMPONENTS,LEVELS,LEADS], 'gradient scales')
    bands = list(dict.fromkeys(vmd['band'].astype(str)))
    if not bands or any(str(b).lower() in ('total','total_mse','all') for b in bands):
        raise ValueError('VMD plotting table must contain frequency contributions and bias, excluding the total.')
    _complete(vmd, ['stage','component','level','band','lead'], [['first','final'],COMPONENTS,LEVELS,bands,LEADS], 'VMD')
    pair_keys = ['component','level','band','lead']
    pair = vmd.pivot(index=pair_keys, columns='stage', values='value')
    pair['value'] = pair['final'] - pair['first']
    paired = pair.reset_index()[pair_keys + ['value']]
    mean_keys = ['stage','component','band','lead']
    mean = vmd.groupby(mean_keys, sort=False, as_index=False)['value'].mean()
    change = paired.groupby(['component','band','lead'], sort=False, as_index=False)['value'].mean()
    check = mean.pivot(index=['component','band','lead'], columns='stage', values='value')
    actual = change.set_index(['component','band','lead'])['value'].reindex(check.index)
    if not np.allclose(actual, check['final'] - check['first'], rtol=1e-12, atol=1e-14):
        raise ValueError('VMD stage difference failed independent pairing verification.')
    final = vmd[vmd.stage == 'final'].copy()
    a72 = _bounds(final.lo72, final.hi72)
    a168 = _bounds(final.lo168, final.hi168)
    positive = (final.lo72.to_numpy(float) > 0) & (final.lo168.to_numpy(float) > 0)
    negative = (final.hi72.to_numpy(float) < 0) & (final.hi168.to_numpy(float) < 0)
    final['ci_status'] = np.where(~(a72 & a168), 'incomplete',
                                  np.where(positive | negative, 'both_exclude_zero_same_sign', 'both_available_other'))
    return {'pinn':pinn, 'gradient_stage':stage, 'gradient_condition':cond,
            'gradient_scales':scales, 'vmd_mean':mean, 'vmd_change':change,
            'vmd_final':final, 'vmd_by_level':vmd, 'bands':bands}


def _limit(values):
    maxval = float(np.max(np.abs(np.asarray(values, float))))
    if maxval == 0:
        return 1.0
    power = 10 ** np.floor(np.log10(maxval))
    return float(np.ceil(maxval / power * 2) / 2 * power)


def _limits(df):
    return {'uv': _limit(df.loc[df.component.isin(['u','v']), 'value']),
            'w': _limit(df.loc[df.component == 'w', 'value'])}


def _unit(comp, mse, exponent):
    unit = (r'\mathrm{Pa}^{2}\,\mathrm{s}^{-2}' if mse else r'\mathrm{Pa}\,\mathrm{s}^{-1}') if comp == 'w' else (
            r'\mathrm{m}^{2}\,\mathrm{s}^{-2}' if mse else r'\mathrm{m}\,\mathrm{s}^{-1}')
    return '$' + (rf'10^{{{exponent}}}\,' if exponent else '') + unit + '$'


def _band_label(value):
    import re
    text = str(value).strip()
    key = re.sub(r'[^a-z0-9]', '', text.lower())
    aliases = {'bias':'Bias', 'biasterm':'Bias', 'dc':'Bias', 'mean':'Bias',
               '28h':'2–8 h', '824h':'8–24 h', '2472h':'24–72 h',
               '72168h':'72–168 h', '24168h':'24–168 h', '168720h':'168–720 h',
               'gt168h':'>168 h', 'gt720h':'>720 h'}
    return aliases.get(key, text.replace('_',' '))


def _matrix(df, ykey, yvalues, column='value'):
    _unique(df, [ykey,'lead'], 'heatmap')
    return df.pivot(index=ykey, columns='lead', values=column).reindex(index=yvalues,columns=LEADS).to_numpy()


def _heat(fig, ax, cax, df, ykey, yvalues, title, comp, limit, mse=False,
          ylabel=None, ci=None, show_y=True, xlabel=True):
    exponent = int(np.floor(np.log10(limit))) if limit < 1 or limit >= 10 else 0
    factor = 10.0 ** exponent
    values = _matrix(df, ykey, yvalues).astype(float) / factor
    edge = limit / factor
    im = ax.imshow(values, cmap=CMAP, norm=TwoSlopeNorm(0, -edge, edge),
                   aspect='auto', interpolation='nearest')
    ax.set_xticks(np.arange(6), LEADS)
    scale_labels = {'raw':'Original','smooth3':'3 × 3 smooth','fine':'Fine remainder'}
    labels = [_band_label(v) if ykey == 'band' else scale_labels.get(v,str(v)) for v in yvalues]
    ax.set_yticks(np.arange(len(yvalues)), labels)
    ax.tick_params(which='both', length=0, pad=4, labelleft=show_y)
    ax.set_xticks(np.arange(-.5,6,1), minor=True)
    ax.set_yticks(np.arange(-.5,len(yvalues),1), minor=True)
    ax.grid(which='minor', color='white', linewidth=1.1)
    if xlabel:
        ax.set_xlabel('Forecast lead (h)', labelpad=5)
    if ylabel:
        ax.set_ylabel(ylabel, labelpad=6)
    ax.set_title(title, loc='left', pad=8)
    for row,col in np.ndindex(values.shape):
        n = values[row,col]
        text = '0' if n == 0 else f'{0 if abs(n)<.005 else n:.2f}'.replace('-', '−')
        ax.text(col,row,text,ha='center',va='center',color='#111111',fontsize=11.5)
    if ci == 'paired':
        lo = _matrix(df,ykey,yvalues,'lower').astype(float)
        hi = _matrix(df,ykey,yvalues,'upper').astype(float)
        avail = _bounds(lo,hi)
        mark = avail & ((lo > 0) | (hi < 0))
        r,c = np.where(mark)
        ax.scatter(c+.33,r-.29,s=12,color='#111111',marker='o',zorder=4)
        r,c = np.where(~avail)
        ax.scatter(c+.33,r-.29,s=17,facecolors='none',edgecolors='#555555',linewidths=.8,marker='s',zorder=4)
    elif ci == 'two_blocks':
        status = _matrix(df,ykey,yvalues,'ci_status')
        r,c = np.where(status == 'both_exclude_zero_same_sign')
        ax.scatter(c+.33,r-.29,s=12,color='#111111',marker='o',zorder=4)
        r,c = np.where(status == 'incomplete')
        ax.scatter(c+.33,r-.29,s=17,facecolors='none',edgecolors='#555555',linewidths=.8,marker='s',zorder=4)
    cb = fig.colorbar(im,cax=cax,ticks=np.linspace(-edge,edge,5))
    cb.ax.tick_params(labelsize=11,length=3,pad=3)
    cb.set_label(_unit(comp,mse,exponent),fontsize=12,labelpad=5)
    return ax


def _row(fig, spec, df, ykey, yvalues, method, start_letter, limits,
         mse=False, ci=None, ylabel=None):
    row = spec.subgridspec(1,3,wspace=.36)
    axes = []
    for j,comp in enumerate(COMPONENTS):
        cell = row[0,j].subgridspec(1,2,width_ratios=[1,.045],wspace=.07)
        ax,cax = fig.add_subplot(cell[0,0]),fig.add_subplot(cell[0,1])
        sub = df[df.component == comp]
        _heat(fig,ax,cax,sub,ykey,yvalues,
              f'({chr(ord(start_letter)+j)}) {method} · {SYMBOLS[comp]}',comp,
              limits['w' if comp=='w' else 'uv'],mse=mse,ci=ci,
              ylabel=ylabel if j==0 else None,show_y=(j==0))
        axes.append(ax)
    return axes


def _pinn_top(fig,spec,df):
    grid = spec.subgridspec(2,2,height_ratios=[1,.16],wspace=.16,hspace=.48)
    axes = [fig.add_subplot(grid[0,j]) for j in range(2)]
    axes[1].sharey(axes[0])
    colors = ['#79A9BD','#E1B47E','#A79BC6']
    for j,seed in enumerate([42,2026]):
        ax=axes[j]; sub=df[df.seed==seed]; x=np.arange(3)
        for term,color,offset in zip(['H','Z','cross'],colors,[-.27,-.09,.09]):
            values=sub[sub.term==term].set_index('level')['value'].reindex(LEVELS).to_numpy(float)
            ax.bar(x+offset,values,width=.165,color=color,edgecolor='white',linewidth=.6,zorder=3)
        total=sub[sub.term=='total'].set_index('level').reindex(LEVELS)
        lo,hi=total.lower.to_numpy(float),total.upper.to_numpy(float); valid=_bounds(lo,hi); xp=x+.29
        ax.vlines(xp[valid],lo[valid],hi[valid],color='#222222',lw=1.5,zorder=5)
        for end in (lo,hi):
            ax.hlines(end[valid],xp[valid]-.045,xp[valid]+.045,color='#222222',lw=1.5,zorder=5)
        ax.scatter(xp,total.value.to_numpy(float),marker='D',s=34,color='#222222',zorder=6)
        ax.axhline(0,color='#777777',lw=.8)
        ax.set_xticks(x,LEVELS); ax.set_xlabel('Pressure level (hPa)',labelpad=5)
        ax.set_title(f'({"ab"[j]}) PINN · Seed {seed} · Stage 1, 1 h',loc='left',pad=8)
        ax.spines[['top','right']].set_visible(False);ax.grid(axis='y',alpha=.2,linewidth=.6)
        ax.yaxis.set_major_locator(MaxNLocator(5));ax.tick_params(labelleft=(j==0))
    finite=df[['value','lower','upper']].to_numpy(float);finite=finite[np.isfinite(finite)]
    low,high=min(0,float(finite.min())),max(0,float(finite.max()));margin=.12*(high-low) or 1
    axes[0].set_ylim(low-margin,high+margin)
    axes[0].set_ylabel(r'PINN $-$ No PINN ($10^{-12}$ s$^{-2}$)',labelpad=8)
    handles=[Patch(facecolor=c,label=l) for c,l in zip(colors,[
        r'$\Delta E[e_H^2]$',r'$\Delta E[e_Z^2]$',r'$\Delta\,2E[e_H e_Z]$'])]
    total = df[df.term=='total']
    complete_ci = _bounds(total.lower,total.upper).all()
    label=r'$\Delta E[e_C^2]$' + (' (95% CI)' if complete_ci else ' (available 95% CI)')
    handles.append(Line2D([0],[0],marker='D',color='#222222',lw=1.2,label=label,markersize=5))
    legend=fig.add_subplot(grid[1,:]);legend.set_axis_off()
    legend.legend(handles=handles,loc='center',ncol=4,frameon=False,fontsize=13,columnspacing=2)


def plot_fig12(p):
    with plt.rc_context(STYLE):
        fig=plt.figure(figsize=(16.6,12.0))
        outer=fig.add_gridspec(3,1,height_ratios=[1.15,.70,1.0],left=.085,right=.925,bottom=.06,top=.965,hspace=.30)
        _pinn_top(fig,outer[0],p['pinn'])
        _row(fig,outer[1],p['gradient_stage'],'level',LEVELS,'Gradient ΔG','c',_limits(p['gradient_stage']),
             ci='paired',ylabel='Pressure (hPa)')
        _row(fig,outer[2],p['vmd_change'],'band',p['bands'],'VMD ΔG','f',_limits(p['vmd_change']),
             mse=True,ylabel='Temporal band')
    return fig


def plot_s16(p):
    with plt.rc_context(STYLE):
        fig=plt.figure(figsize=(16.6,4.2))
        outer=fig.add_gridspec(1,1,left=.085,right=.925,bottom=.20,top=.86)
        _row(fig,outer[0],p['gradient_condition'],'level',LEVELS,'High − low','a',_limits(p['gradient_condition']),
             ci='paired',ylabel='Pressure (hPa)')
    return fig


def plot_s17(p):
    df=p['vmd_mean'];df=df[df.stage=='final'];limits=_limits(df)
    with plt.rc_context(STYLE):
        fig=plt.figure(figsize=(16.6,5.2))
        outer=fig.add_gridspec(1,1,left=.085,right=.925,bottom=.16,top=.88)
        _row(fig,outer[0],df,'band',p['bands'],'Final forecast','a',limits,mse=True,ylabel='Temporal band')
    return fig


def plot_s18(p):
    df=p['gradient_scales'];limits=_limits(df)
    with plt.rc_context(STYLE):
        fig=plt.figure(figsize=(16.6,11.4))
        outer=fig.add_gridspec(3,1,left=.085,right=.925,bottom=.065,top=.96,hspace=.38)
        for r,(scale,label) in enumerate([('raw','Original'),('smooth3','3 × 3 smooth'),('fine','Fine remainder')]):
            _row(fig,outer[r],df[df.scale==scale],'level',LEVELS,label,chr(97+3*r),limits,ci='paired',ylabel='Pressure (hPa)')
    return fig


def plot_s19(p):
    df=p['vmd_final'];limits=_limits(df)
    with plt.rc_context(STYLE):
        fig=plt.figure(figsize=(16.6,12.8))
        outer=fig.add_gridspec(3,1,left=.085,right=.925,bottom=.06,top=.965,hspace=.30)
        for r,level in enumerate(LEVELS):
            _row(fig,outer[r],df[df.level==level],'band',p['bands'],f'{level} hPa',chr(97+3*r),limits,
                 mse=True,ci='two_blocks',ylabel='Temporal band')
    return fig


FIGURE_BUILDERS={'Fig12':plot_fig12,'FigS16':plot_s16,'FigS17':plot_s17,'FigS18':plot_s18,'FigS19':plot_s19}


In [ ]:
from PIL import Image
plot_data = load_plot_data(CONFIG, memory=MEMORY_TABLES)
prepared = prepare_figures(plot_data)
for stem, builder in FIGURE_BUILDERS.items():
    fig = builder(prepared)
    fig.savefig(FIGURES / f'{stem}.png', dpi=DPI, facecolor='white')
    with Image.open(FIGURES / f'{stem}.png') as image:
        image.convert('RGB').save(FIGURES / f'{stem}.tiff', compression='tiff_lzw', dpi=(DPI, DPI))
    
    plt.close(fig)
